# Nettoyage des données logistiques

Une IA générative (Claude) a été utilisée pour relire et corriger ce nettoyage et ajouter des commentaires. 

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

## 1. Exploration des données

In [2]:
data_dictionary = pd.read_csv("../data/data_dictionary.csv")
data_dictionary

,source_bundle,source_system_id,file_name,field_name,logical_type,key_role,description,example_value
0,SB02,SYS-CLN-OPS,clinical_encounters.csv,encounter_number,identifier,primary_or_composite_key,Source-local encounter identifier.,ENC-000001
1,SB02,SYS-CLN-OPS,clinical_encounters.csv,patient_number,identifier,non_key,Source-local patient identifier used within th...,PAT-31162
2,SB02,SYS-CLN-OPS,clinical_encounters.csv,encounter_start,date_or_datetime,non_key,Encounter start.,2018-02-27T09:15:00
3,SB02,SYS-CLN-OPS,clinical_encounters.csv,encounter_end,date_or_datetime,non_key,Encounter end.,2018-02-27T10:45:00
4,SB02,SYS-CLN-OPS,clinical_encounters.csv,facility_license_number,identifier,non_key,Facility license number.,FAC-VLN-101
...,...,...,...,...,...,...,...,...
350,SB06,SYS-DOC-PUB,public_and_investigative_documents.csv,named_location,categorical_or_text,non_key,Named location.,Valenne Central District
351,SB06,SYS-DOC-PUB,public_and_investigative_documents.csv,title,categorical_or_text,non_key,Title.,Restricted Review of Recurrent Executive Healt...
352,SB06,SYS-DOC-PUB,public_and_investigative_documents.csv,text,long_text,non_key,Full document body as stored by the source rep...,PURPOSE\r\nThis restricted briefing summarizes...
353,SB06,SYS-DOC-PUB,public_and_investigative_documents.csv,public_or_restricted,categorical_or_text,non_key,Public or restricted.,restricted


In [3]:

domaines = ["public_records", "health", "ecology", "logistics"]
dataframes = {}

for domaine in domaines:
    data_dir = Path(f"../data/{domaine}")
    csv_files = list(data_dir.glob("*.csv"))
    for file in csv_files:
        name = file.stem
        dataframes[name] = pd.read_csv(file)
        print(f"--- {domaine} / {name} ---")
        print(dataframes[name].shape)
        print(dataframes[name].dtypes)
        print(dataframes[name].isna().sum())
        print()

--- public_records / institutional_memos ---
(30, 12)
document_id                object
document_date              object
document_type              object
author_name                object
author_organization        object
recipient_text             object
subject_line               object
text                       object
related_purchase_order     object
related_shipment_number    object
related_case_number        object
visibility_level           object
dtype: object
document_id                 0
document_date               0
document_type               0
author_name                 0
author_organization         0
recipient_text              0
subject_line                0
text                        0
related_purchase_order     25
related_shipment_number    22
related_case_number        20
visibility_level            0
dtype: int64

--- public_records / licensed_facilities ---
(3, 12)
facility_license_number                 object
facility_name                           object
fac

### fonction d'analyse


In [4]:
# Formats de dates rencontrés dans les fichiers bruts.
# Les dates avec "/" et l'année à la fin sont au format jour/mois/année :
# "30/06/2025" (goods_receipts) ou "16/02/2018 12:00:00" (shipments) ne peuvent pas être lues autrement.
FORMATS_DATE = [
    "%Y-%m-%d", "%Y-%m-%dT%H:%M:%S", "%Y-%m-%d %H:%M:%S",
    "%Y/%m/%d", "%Y/%m/%d %H:%M:%S",
    "%d/%m/%Y", "%d/%m/%Y %H:%M:%S",
    "%d-%b-%Y", "%d-%b-%Y %H:%M:%S",
]

def parser_dates_robuste(serie):
    if pd.api.types.is_datetime64_any_dtype(serie):
        return serie  # déjà au format date

    # On convertit tout en texte propre en évitant le texte "nan" ou "NaT"
    s = serie.astype("string").str.strip()
    s = s.replace(["nan", "NaT", "None", ""], pd.NA)

    # On essaie chaque format un par un (avec format="mixed", "12/07/2024" serait lu comme le 7 décembre)
    resultat = pd.Series(pd.NaT, index=serie.index, dtype="datetime64[ns]")
    for fmt in FORMATS_DATE:
        a_lire = resultat.isna() & s.notna()
        if not a_lire.any():
            break
        resultat[a_lire] = pd.to_datetime(s[a_lire], format=fmt, errors="coerce")

    non_lues = s.notna() & resultat.isna()
    if non_lues.any():
        print(f"Attention : {non_lues.sum()} date(s) non reconnue(s) : {s[non_lues].unique()[:5].tolist()}")
    return resultat

In [5]:
def valeurs_non_numeriques(df, colonne):
    non_num = pd.to_numeric(df[colonne], errors="coerce").isna() & df[colonne].notna()
    return df.loc[non_num, [colonne]].value_counts()

In [6]:
def str_to_num(nom_table, colonne):
    s = dataframes[nom_table][colonne].astype(str)
    # On extrait le premier nombre de la cellule, en gardant le signe "-" :
    # "-2.9 case" -> -2.9 ; "56,41" -> 56.41 ; "14 kit" -> 14
    s_nettoye = s.str.extract(r'(-?\d+(?:[.,]\d+)?)', expand=False).str.replace(',', '.', regex=False)
    dataframes[nom_table][colonne] = pd.to_numeric(s_nettoye, errors='coerce').astype(float)

In [7]:
def vue_rapide_complete(df, nom_fichier_csv, dictionnaire):
    print(f"====== Analyse complète de {nom_fichier_csv} ======")
    print(f"Taille : {df.shape[0]} lignes, {df.shape[1]} colonnes")
    print(f"Lignes en double : {df.duplicated().sum()}\n")
    
    # Bilan global des valeurs manquantes
    trous = df.isna().sum()
    colonnes_avec_trous = trous[trous > 0]
    if len(colonnes_avec_trous) > 0:
        print("[Trous détectés] Valeurs manquantes (NaN) par colonne :")
        print(colonnes_avec_trous)
        print()

    regles_dic = dictionnaire[dictionnaire["file_name"] == nom_fichier_csv]
    
    if len(regles_dic) == 0:
        print(f"ERREUR : Aucun enregistrement trouvé pour '{nom_fichier_csv}' dans le dictionnaire.\n")
        return
        
    print("--- Vérification par type logique (Dictionnaire vs DataFrame) ---")
    
    for _, ligne in regles_dic.iterrows():
        col = ligne["field_name"]
        type_attendu = ligne["logical_type"]
        role = ligne["key_role"]
        
        if col in df.columns:
            dtype_actuel = str(df[col].dtype)
            
            # Vérification des types mélangés dans la colonne
            # On ignore les NaN (NoneType) pour ne compter que les vraies valeurs
            types_presents = df[col].dropna().apply(type).nunique()
            if types_presents > 1:
                details_types = df[col].dropna().apply(type).value_counts().to_dict()
                print(f"[Types mixtes] '{col}' contient plusieurs types différents : {details_types}")
            
            # Vérification des colonnes numériques
            if type_attendu == "numeric":
                if not pd.api.types.is_numeric_dtype(df[col]):
                    print(f"[Numérique] '{col}' est lu comme du texte ({dtype_actuel}) -> À nettoyer !")
                else:
                    print(f"[Numérique] '{col}' : Format nombre OK.")
            
            # Vérification des dates
            elif type_attendu == "date_or_datetime":
                if not pd.api.types.is_datetime64_any_dtype(df[col]):
                    print(f"[Date] '{col}' est lu comme du texte ({dtype_actuel}) -> À convertir !")
                else:
                    print(f"[Date] '{col}' : Format date OK.")
            
            # Vérification des clés primaires
            if role == "primary_or_composite_key":
                nb_doublons_cle = df[col].duplicated().sum()
                if nb_doublons_cle > 0:
                    print(f"[Clé primaire] '{col}' contient {nb_doublons_cle} doublons !")
                else:
                    print(f"[Clé primaire] '{col}' : Unique et valide.")
        else:
            print(f"La colonne '{col}' listée dans le dictionnaire est absente du DataFrame.")
            
    print("\n")
    print("==================================================\n")

In [8]:
import re

def normaliser_texte(df, colonnes_categorielles):
    # 1) Toutes les colonnes texte : on enlève les espaces en début/fin et les espaces multiples
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]) or pd.api.types.is_datetime64_any_dtype(df[col]):
            continue
        df[col] = df[col].map(lambda v: re.sub(r"\s+", " ", v).strip() if isinstance(v, str) else v)

    # 2) Colonnes catégorielles : "POSTED" et "Posted" deviennent la graphie la plus fréquente ("posted")
    for col in colonnes_categorielles:
        cle = df[col].map(lambda v: v.lower() if isinstance(v, str) else v)
        graphie_majoritaire = df[col].groupby(cle).agg(lambda x: x.value_counts().index[0])
        df[col] = cle.map(graphie_majoritaire)
    return df

#### Nettoyage de erp_business_partners

Problèmes : cellules vides ou remplies d'espaces, date `onboarding_date` lue comme du texte, une région de facturation manquante. Corrections : vides → NaN, conversion de la date, région déduite de la devise (chaque devise correspond à un seul pays).

In [9]:
vue_rapide_complete(dataframes["erp_business_partners"], "erp_business_partners.csv", data_dictionary)


====== Analyse complète de erp_business_partners.csv ======
Taille : 80 lignes, 11 colonnes
Lignes en double : 0

[Trous détectés] Valeurs manquantes (NaN) par colonne :
external_registration_number    14
billing_region                   1
dtype: int64

--- Vérification par type logique (Dictionnaire vs DataFrame) ---
[Clé primaire] 'partner_code' : Unique et valide.
[Date] 'onboarding_date' est lu comme du texte (object) -> À convertir !





In [10]:
# 1. Remplacer les cellules qui contiennent uniquement des espaces (vides en apparence) par des NaN
dataframes["erp_business_partners"].replace(r'^\s*$', np.nan, regex=True, inplace=True)

# 2. Remplacer des mots-clés spécifiques par de vrais NaN
dataframes["erp_business_partners"].replace(["N/A", "-", "Inconnu", "null", ""], np.nan, inplace=True)

In [11]:
# 1. Conversion de la date (même fonction que pour les autres tables : "01/10/2003" = 1er octobre 2003)
dataframes["erp_business_partners"]["onboarding_date"] = parser_dates_robuste(
    dataframes["erp_business_partners"]["onboarding_date"]
)

In [12]:
# 2. Combler le trou de 'billing_region'.
# 'shipping_region' n'est pas au même niveau géographique ("Kintara Karst Cluster" est une région
# à l'intérieur de "Kintara Republic") : l'utiliser créerait une modalité de pays qui n'existe pas.
# En revanche, sur les 79 partenaires renseignés, chaque devise correspond à un seul pays de facturation
# (KRN -> Kintara Republic, VLN -> Valenne Republic, ...). On apprend cette correspondance sur les données
# et on l'utilise pour compléter la valeur manquante (le préfixe du numéro d'enregistrement, ex. REG-KNT, la confirme).
bp = dataframes["erp_business_partners"]
connus = bp.dropna(subset=["billing_region"])
pays_par_devise = connus.groupby("default_currency")["billing_region"].agg(lambda s: s.unique().tolist())
print(pays_par_devise)
assert pays_par_devise.map(len).eq(1).all(), "Une devise correspond à plusieurs pays : imputation non fiable"

manquants = bp["billing_region"].isna()
bp.loc[manquants, "billing_region"] = bp.loc[manquants, "default_currency"].map(pays_par_devise.str[0])
display(bp.loc[manquants, ["partner_code", "external_registration_number", "default_currency", "shipping_region", "billing_region"]])

default_currency
AVL      [Avelora Federation]
KRN        [Kintara Republic]
NDH         [Nordhaven Union]
TMB    [Tembara Commonwealth]
VLN        [Valenne Republic]
Name: billing_region, dtype: object


,partner_code,external_registration_number,default_currency,shipping_region,billing_region
28,BP-30029,REG-KNT-20014,KRN,Kintara Karst Cluster,Kintara Republic


#### Nettoyage de erp_facilities

Problèmes : date `opening_date` lue comme du texte, une valeur manquante dans `storage_capabilities`. Corrections : conversion de la date ; la valeur manquante est laissée vide, rien ne permet de la deviner.


In [13]:
vue_rapide_complete(dataframes["erp_facilities"], "erp_facilities.csv", data_dictionary)

====== Analyse complète de erp_facilities.csv ======
Taille : 30 lignes, 11 colonnes
Lignes en double : 0

[Trous détectés] Valeurs manquantes (NaN) par colonne :
storage_capabilities    1
dtype: int64

--- Vérification par type logique (Dictionnaire vs DataFrame) ---
[Clé primaire] 'facility_code' : Unique et valide.
[Numérique] 'latitude' : Format nombre OK.
[Numérique] 'longitude' : Format nombre OK.
[Date] 'opening_date' est lu comme du texte (object) -> À convertir !





In [14]:
# 1. Remplacer les cellules qui contiennent uniquement des espaces (vides en apparence) par des NaN
dataframes["erp_facilities"].replace(r'^\s*$', np.nan, regex=True, inplace=True)

# 2. Remplacer des mots-clés spécifiques par de vrais NaN
dataframes["erp_facilities"].replace(["N/A", "-", "Inconnu", "null", ""], np.nan, inplace=True)

In [15]:
dataframes["erp_facilities"]["opening_date"] = parser_dates_robuste(dataframes["erp_facilities"]["opening_date"])


#### Nettoyage de goods_receipts

Problèmes : 2 lignes en double (même `receipt_number`), dates dans plusieurs formats (dont jour/mois/année), quantités reçues stockées en texte. Corrections : suppression des doublons, dates uniformisées, quantités converties en nombres.

In [16]:
vue_rapide_complete(dataframes["goods_receipts"], "goods_receipts.csv", data_dictionary)

====== Analyse complète de goods_receipts.csv ======
Taille : 612 lignes, 12 colonnes
Lignes en double : 2

[Trous détectés] Valeurs manquantes (NaN) par colonne :
inspection_status    4
dtype: int64

--- Vérification par type logique (Dictionnaire vs DataFrame) ---
[Clé primaire] 'receipt_number' contient 2 doublons !
[Date] 'receipt_date' est lu comme du texte (object) -> À convertir !
[Numérique] 'received_quantity' est lu comme du texte (object) -> À nettoyer !





In [17]:
# 1. Remplacer les cellules qui contiennent uniquement des espaces (vides en apparence) par des NaN
dataframes["goods_receipts"].replace(r'^\s*$', np.nan, regex=True, inplace=True)

# 2. Remplacer des mots-clés spécifiques par de vrais NaN
dataframes["goods_receipts"].replace(["N/A", "-", "Inconnu", "null", ""], np.nan, inplace=True)

In [18]:
#correction de la date
dataframes["goods_receipts"]["receipt_date"] = parser_dates_robuste(dataframes["goods_receipts"]["receipt_date"])

In [19]:
#suppresion des doublons
dataframes["goods_receipts"] = dataframes["goods_receipts"].drop_duplicates(subset=['receipt_number']).copy()

In [20]:
#correction de type
str_to_num("goods_receipts","received_quantity")

#### Nettoyage de inventory_adjustments

Problèmes : date lue comme du texte, quantités stockées en texte avec une unité. Corrections : conversion de la date et des quantités **en gardant le signe** (un ajustement négatif est un retrait de stock).

In [21]:
vue_rapide_complete(dataframes["inventory_adjustments"], "inventory_adjustments.csv", data_dictionary)

====== Analyse complète de inventory_adjustments.csv ======
Taille : 140 lignes, 11 colonnes
Lignes en double : 0

[Trous détectés] Valeurs manquantes (NaN) par colonne :
approved_by_role             1
related_receipt_number     117
related_shipment_number    103
dtype: int64

--- Vérification par type logique (Dictionnaire vs DataFrame) ---
[Clé primaire] 'adjustment_number' : Unique et valide.
[Date] 'adjustment_date' est lu comme du texte (object) -> À convertir !
[Numérique] 'adjustment_quantity' est lu comme du texte (object) -> À nettoyer !





In [22]:
#correction de la date
dataframes["inventory_adjustments"]["adjustment_date"] = parser_dates_robuste(dataframes["inventory_adjustments"]["adjustment_date"])

In [23]:
#correction de type
str_to_num("inventory_adjustments","adjustment_quantity")

In [24]:
# 1. Remplacer les cellules qui contiennent uniquement des espaces (vides en apparence) par des NaN
dataframes["inventory_adjustments"].replace(r'^\s*$', np.nan, regex=True, inplace=True)

# 2. Remplacer des mots-clés spécifiques par de vrais NaN
dataframes["inventory_adjustments"].replace(["N/A", "-", "Inconnu", "null", ""], np.nan, inplace=True)

#### Nettoyage de inventory_month_end

La clé est composite (site + mois + catégorie) : les « doublons » signalés colonne par colonne sont normaux. Sur la clé complète, 2 lignes en double sont supprimées. Les mois sont convertis en dates et les quantités en nombres.

In [25]:
vue_rapide_complete(dataframes["inventory_month_end"], "inventory_month_end.csv", data_dictionary)

====== Analyse complète de inventory_month_end.csv ======
Taille : 502 lignes, 11 colonnes
Lignes en double : 2

--- Vérification par type logique (Dictionnaire vs DataFrame) ---
[Clé primaire] 'facility_code' contient 497 doublons !
[Clé primaire] 'reporting_month' contient 482 doublons !
[Clé primaire] 'item_category' contient 497 doublons !
[Numérique] 'opening_stock' est lu comme du texte (object) -> À nettoyer !
[Numérique] 'received_quantity' est lu comme du texte (object) -> À nettoyer !
[Numérique] 'outgoing_quantity' est lu comme du texte (object) -> À nettoyer !
[Numérique] 'documented_internal_use' : Format nombre OK.
[Numérique] 'documented_waste' est lu comme du texte (object) -> À nettoyer !
[Numérique] 'closing_stock' est lu comme du texte (object) -> À nettoyer !





In [26]:
dataframes["inventory_month_end"]["reporting_month"] = parser_dates_robuste(
    dataframes["inventory_month_end"]["reporting_month"]
)

In [27]:
cles = ['facility_code', 'reporting_month', 'item_category']
print("Doublons sur la clé composite :", dataframes["inventory_month_end"].duplicated(subset=cles).sum())

df_month = dataframes["inventory_month_end"]

# Affiche toutes les lignes impliquées dans les doublons
lignes_en_conflit = df_month[df_month.duplicated(subset=cles, keep=False)]
display(lignes_en_conflit.sort_values(by=cles))



Doublons sur la clé composite : 2


,facility_code,reporting_month,item_category,opening_stock,received_quantity,outgoing_quantity,documented_internal_use,documented_waste,closing_stock,unit,inventory_status
170,FCL-1019,2021-05-01,protective_equipment,57.10,11.93,0.00,1.03,2.10,64.71,case,closed
501,FCL-1019,2021-05-01,protective_equipment,57.10,11.93,0.00,1.03,2.10,64.71,case,closed
284,FCL-1020,2019-08-01,agricultural_product,43.82,10.93,0.00,3.98,0.94,51.14,pallet,closed
500,FCL-1020,2019-08-01,agricultural_product,43.82,10.93,0.00,3.98,0.94,51.14,pallet,closed


In [28]:
dataframes["inventory_month_end"] = df_month.drop_duplicates(subset=cles, keep='first').copy()

In [29]:
#correction de type
str_to_num("inventory_month_end","opening_stock")
str_to_num("inventory_month_end","received_quantity")
str_to_num("inventory_month_end","outgoing_quantity")
str_to_num("inventory_month_end","documented_waste")
str_to_num("inventory_month_end","closing_stock")

#### Nettoyage de purchase_order_lines

Clé composite (numéro de commande + ligne) : 1 doublon supprimé. Dates converties, quantités et prix convertis en nombres (virgules décimales, unités collées).

In [30]:
vue_rapide_complete(dataframes["purchase_order_lines"], "purchase_order_lines.csv", data_dictionary)

====== Analyse complète de purchase_order_lines.csv ======
Taille : 651 lignes, 16 colonnes
Lignes en double : 1

[Trous détectés] Valeurs manquantes (NaN) par colonne :
declared_cost_center    3
dtype: int64

--- Vérification par type logique (Dictionnaire vs DataFrame) ---
[Clé primaire] 'purchase_order_number' contient 1 doublons !
[Clé primaire] 'purchase_order_line' contient 650 doublons !
[Date] 'order_date' est lu comme du texte (object) -> À convertir !
[Numérique] 'ordered_quantity' est lu comme du texte (object) -> À nettoyer !
[Numérique] 'unit_price' est lu comme du texte (object) -> À nettoyer !
[Date] 'requested_delivery_date' est lu comme du texte (object) -> À convertir !





In [31]:
cles = ['purchase_order_number', 'purchase_order_line']
print("Doublons sur la clé composite :", dataframes["purchase_order_lines"].duplicated(subset=cles).sum())

df_po = dataframes["purchase_order_lines"]

# Affiche toutes les lignes impliquées dans les doublons
lignes_en_conflit = df_po[df_po.duplicated(subset=cles, keep=False)]
display(lignes_en_conflit.sort_values(by=cles))

Doublons sur la clé composite : 1


,purchase_order_number,purchase_order_line,order_date,ordering_facility_code,supplier_partner_code,item_code,item_description,item_category,ordered_quantity,ordered_unit,unit_price,currency,requested_delivery_date,authorization_class,declared_cost_center,order_status
348,PO-2022-00022,1,2022-02-09,FCL-1019,BP-30016,ITM-CSE-002,Insulated transport container,cold_storage_equipment,21,unit,3052.22,AVL,2022-03-17,ordinary,Facilities,completed
650,PO-2022-00022,1,2022-02-09,FCL-1019,BP-30016,ITM-CSE-002,Insulated transport container,cold_storage_equipment,21,unit,3052.22,AVL,2022-03-17,ordinary,Facilities,completed


In [32]:
dataframes["purchase_order_lines"] = df_po.drop_duplicates(subset=cles, keep='first').copy()

In [33]:
#correction de la date
dataframes["purchase_order_lines"]["order_date"] = parser_dates_robuste(dataframes["purchase_order_lines"]["order_date"])
dataframes["purchase_order_lines"]["requested_delivery_date"] = parser_dates_robuste(dataframes["purchase_order_lines"]["requested_delivery_date"])

In [34]:
#correction de type
str_to_num("purchase_order_lines","ordered_quantity")
str_to_num("purchase_order_lines","unit_price")

In [35]:
# 1. Remplacer les cellules qui contiennent uniquement des espaces (vides en apparence) par des NaN
dataframes["purchase_order_lines"].replace(r'^\s*$', np.nan, regex=True, inplace=True)

# 2. Remplacer des mots-clés spécifiques par de vrais NaN
dataframes["purchase_order_lines"].replace(["N/A", "-", "Inconnu", "null", ""], np.nan, inplace=True)

#### Nettoyage de shipments

Clé composite (numéro d'expédition + étape) : 2 doublons supprimés. Dates de départ et d'arrivée converties (plusieurs formats) ; les 17 dates d'arrivée manquantes sont laissées vides.

In [36]:
vue_rapide_complete(dataframes["shipments"], "shipments.csv", data_dictionary)

====== Analyse complète de shipments.csv ======
Taille : 452 lignes, 16 colonnes
Lignes en double : 2

[Trous détectés] Valeurs manquantes (NaN) par colonne :
arrival_time    17
dtype: int64

--- Vérification par type logique (Dictionnaire vs DataFrame) ---
[Clé primaire] 'shipment_number' contient 2 doublons !
[Clé primaire] 'shipment_leg' contient 451 doublons !
[Date] 'dispatch_time' est lu comme du texte (object) -> À convertir !
[Date] 'arrival_time' est lu comme du texte (object) -> À convertir !
[Numérique] 'declared_quantity' : Format nombre OK.





In [37]:
cles = ['shipment_number', 'shipment_leg']
print("Doublons sur la clé composite :", dataframes["shipments"].duplicated(subset=cles).sum())

df_exp = dataframes["shipments"]

# Affiche toutes les lignes impliquées dans les doublons
lignes_en_conflit = df_exp[df_exp.duplicated(subset=cles, keep=False)]
display(lignes_en_conflit.sort_values(by=cles))

Doublons sur la clé composite : 2


,shipment_number,shipment_leg,dispatch_time,arrival_time,sender_partner_code,receiver_partner_code,origin_facility_code,destination_facility_code,carrier_partner_code,item_category,declared_quantity,declared_unit,temperature_control_class,authorization_class,declared_purpose,shipment_status
26,SHP-2016-00009,1,2016-09-05T13:00:00,2016-09-08T02:00:00,BP-30043,BP-30036,FCL-1010,FCL-1030,BP-30020,consulting_service,1,service,ambient,emergency,scheduled stock transfer,delivered
451,SHP-2016-00009,1,2016-09-05T13:00:00,2016-09-08T02:00:00,BP-30043,BP-30036,FCL-1010,FCL-1030,BP-30020,consulting_service,1,service,ambient,emergency,scheduled stock transfer,delivered
251,SHP-2022-00017,1,2022-02-16T14:00:00,2022-02-18T15:00:00,BP-30073,BP-30078,FCL-1019,FCL-1011,BP-30031,protective_equipment,3,case,ambient,ordinary,agricultural distribution,delivered
450,SHP-2022-00017,1,2022-02-16T14:00:00,2022-02-18T15:00:00,BP-30073,BP-30078,FCL-1019,FCL-1011,BP-30031,protective_equipment,3,case,ambient,ordinary,agricultural distribution,delivered


In [38]:
dataframes["shipments"] = df_exp.drop_duplicates(subset=cles, keep='first').copy()

In [39]:
#correction de la date
dataframes["shipments"]["dispatch_time"] = parser_dates_robuste(dataframes["shipments"]["dispatch_time"])
dataframes["shipments"]["arrival_time"] = parser_dates_robuste(dataframes["shipments"]["arrival_time"])

#### Harmonisation des textes et des catégories (toutes les tables)

Certaines colonnes catégorielles contiennent la même modalité écrite de plusieurs façons (`posted` / `POSTED`, `2-8C` / `2-8c`, `transfer posting correction` / `TRANSFER POSTING CORRECTION`) ou avec des espaces en trop (`"  scheduled  stock  transfer "`). Sans correction, ces variantes sont comptées comme des catégories différentes : elles deviennent des modalités « rares » artificielles pour un encodage one-hot, un clustering ou Apriori.

On supprime donc les espaces superflus dans toutes les colonnes texte, puis, pour chaque colonne catégorielle, on remplace chaque variante par la graphie la plus fréquente de la modalité. Les identifiants (codes) et les noms propres ne changent pas de casse.

In [40]:
colonnes_categorielles = {
    "goods_receipts": ["received_unit", "inspection_status", "receipt_status"],
    "inventory_adjustments": ["item_category", "unit", "adjustment_reason", "approved_by_role", "adjustment_status"],
    "inventory_month_end": ["item_category", "unit", "inventory_status"],
    "purchase_order_lines": ["item_category", "ordered_unit", "currency", "authorization_class",
                             "declared_cost_center", "order_status"],
    "shipments": ["item_category", "declared_unit", "temperature_control_class", "authorization_class",
                  "declared_purpose", "shipment_status"],
    "erp_facilities": ["facility_category", "region", "facility_status"],
    "erp_business_partners": ["partner_role", "billing_region", "shipping_region", "default_currency",
                              "payment_terms", "partner_status", "risk_class"],
}

for nom_table, colonnes in colonnes_categorielles.items():
    df = dataframes[nom_table]
    avant = {col: df[col].nunique() for col in colonnes}
    normaliser_texte(df, colonnes)
    for col in colonnes:
        apres = df[col].nunique()
        if apres != avant[col]:
            print(f"{nom_table}.{col} : {avant[col]} -> {apres} modalités")

goods_receipts.receipt_status : 3 -> 2 modalités
inventory_adjustments.adjustment_reason : 12 -> 10 modalités
inventory_month_end.inventory_status : 5 -> 3 modalités
purchase_order_lines.item_category : 23 -> 18 modalités
shipments.temperature_control_class : 6 -> 3 modalités
shipments.declared_purpose : 12 -> 8 modalités
shipments.shipment_status : 7 -> 4 modalités
erp_business_partners.partner_role : 6 -> 5 modalités


In [41]:

# 1. Dossier de sortie
dossier_sortie = Path("../data/logistics/clean")
dossier_sortie.mkdir(parents=True, exist_ok=True)
compteur = 0

# 2. Les 7 tables logistiques à exporter
tables_logistiques = [
    "goods_receipts",
    "inventory_adjustments",
    "inventory_month_end",
    "purchase_order_lines",
    "shipments",
    "erp_facilities",
    "erp_business_partners"
]

for nom_table in tables_logistiques:
    if nom_table in dataframes:
        chemin_complet = dossier_sortie / f"{nom_table}_clean.csv"
        dataframes[nom_table].to_csv(chemin_complet, index=False, na_rep="NaN")
        compteur += 1
        print(f"Exporté : {nom_table}_clean.csv")

print(f"\n{compteur} fichiers exportés dans : {dossier_sortie}")

Exporté : goods_receipts_clean.csv
Exporté : inventory_adjustments_clean.csv
Exporté : inventory_month_end_clean.csv
Exporté : purchase_order_lines_clean.csv
Exporté : shipments_clean.csv
Exporté : erp_facilities_clean.csv
Exporté : erp_business_partners_clean.csv

7 fichiers exportés dans : data\logistics\clean


In [42]:
tables_logistiques = [
    "goods_receipts",
    "inventory_adjustments",
    "inventory_month_end",
    "purchase_order_lines",
    "shipments",
    "erp_facilities",
    "erp_business_partners"
]

for nom_table in tables_logistiques:
    vue_rapide_complete(dataframes[nom_table], nom_table +".csv", data_dictionary)

====== Analyse complète de goods_receipts.csv ======
Taille : 610 lignes, 12 colonnes
Lignes en double : 0

[Trous détectés] Valeurs manquantes (NaN) par colonne :
inspection_status    4
dtype: int64

--- Vérification par type logique (Dictionnaire vs DataFrame) ---
[Clé primaire] 'receipt_number' : Unique et valide.
[Date] 'receipt_date' : Format date OK.
[Numérique] 'received_quantity' : Format nombre OK.



====== Analyse complète de inventory_adjustments.csv ======
Taille : 140 lignes, 11 colonnes
Lignes en double : 0

[Trous détectés] Valeurs manquantes (NaN) par colonne :
approved_by_role             1
related_receipt_number     117
related_shipment_number    103
dtype: int64

--- Vérification par type logique (Dictionnaire vs DataFrame) ---
[Clé primaire] 'adjustment_number' : Unique et valide.
[Date] 'adjustment_date' : Format date OK.
[Numérique] 'adjustment_quantity' : Format nombre OK.



====== Analyse complète de inventory_month_end.csv ======
Taille : 500 lignes, 11 colon